# Frame classifier

Turns the frame features into calibrated probabilities `P(H)`, `P(A)` and `P(D)` for every frame.

* `WindowFeatureBuilder` adds context around each frame, per half: rolling statistics and lead / lag differences over
  +/-1 s, means and trends over 4 s and 12 s, and trends over 30 s for team shape.
* `PossessionClassifier` is a multiclass CatBoost model. `fit` trains it on the labelled external matches
  (`helper_data.ipynb`) with early stopping on two held-out matches.
* `save_bundle`, `load_bundle` and `predict_bundle` keep the model in `models/<name>/` together with what it needs:
  feature columns, temperature, decoder settings, feature and preprocessing configs. You can then label another match,
  or score against another ground truth, without retraining (`helper_predict.ipynb`).

The model never sees the absolute ball x / y, so it cannot memorise where on the pitch things happened in a given match.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_features.ipynb", globals())
from scipy.optimize import minimize_scalar
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, log_loss

In [ ]:
# Never model inputs: absolute pitch coordinates (would let the model memorise WHERE things happened).
MODEL_EXCLUDE = ("ball_x", "ball_y")


def model_view(F: pd.DataFrame) -> pd.DataFrame:
    return F.drop(columns=[c for c in F.columns if c in MODEL_EXCLUDE])


def _roll(a: np.ndarray, lo: int, hi: int, want_std: bool = False, min_frac: float = 0.2):
    """NaN-aware mean (and std) of every column over the frames [t+lo, t+hi] around each frame t (window clipped to the array).
    NaN when fewer than ``min_frac`` of the window's frames are valid. Cumulative sums, so cost does not depend on the window length."""
    n, k = a.shape
    ok = np.isfinite(a)
    v = np.where(ok, a, 0.0)
    z = np.zeros((1, k))
    cs, cc = np.vstack([z, np.cumsum(v, axis=0)]), np.vstack([z, np.cumsum(ok, axis=0)])
    t = np.arange(n)
    i0, i1 = np.clip(t + lo, 0, n), np.clip(t + hi + 1, 0, n)
    cnt = cc[i1] - cc[i0]
    good = cnt >= max(1.0, min_frac * (hi - lo + 1))
    mean = np.where(good, (cs[i1] - cs[i0]) / np.maximum(cnt, 1), np.nan)
    if not want_std:
        return mean, None
    cq = np.vstack([z, np.cumsum(v * v, axis=0)])
    var = (cq[i1] - cq[i0]) / np.maximum(cnt, 1) - mean ** 2
    return mean, np.where(good, np.sqrt(np.maximum(var, 0.0)), np.nan)


class WindowFeatureBuilder:
    """Context around every frame at three ranges. Computed per half; offline, so the future is available as well as the past.

    short   (+/- 1 s)   rolling mean / std / min / max and 0.5 s / 1 s lead / lag differences of the cues that change fastest
    medium  (4 s, 12 s) mean over the centred window, and trend = mean of the next N s minus mean of the previous N s
    long    (30 s)      trend only, for the slow team-shape columns
    The medium / long ranges carry the information through the long missing-ball stretches (the ball is missing 41% of BAR-ATH)."""
    KEY_COLUMNS = ["delta_d", "d_min", "d_H1", "d_A1", "nearest_is_home", "soft_home_share_2m", "soft_home_share_5m",
                   "home_share_5m", "home_share_10m", "ball_speed", "ball_in_pitch", "ball_out_dist", "ball_observed",
                   "frac_still", "ball_anomaly", "nbr_H_15m", "nbr_A_15m", "nearH_closing", "nearA_closing", "mean_speed",
                   "nearH_rel_speed", "nearA_rel_speed"]
    CONTEXT_COLUMNS = ["delta_d", "nearest_is_home", "soft_home_share_5m", "d_min", "ball_speed", "ball_in_pitch", "ball_observed",
                       "frac_still", "mean_speed", "nearH_rel_speed", "nearA_rel_speed", "press_on_H1", "press_on_A1",
                       "support_H1", "support_A1", "team_fwd_vel_H", "team_fwd_vel_A", "team_speed_H", "team_speed_A",
                       "ball_minus_cent_H", "cent_A_minus_ball", "ball_x_hp", "H_ahead_of_ball", "A_ahead_of_ball"]
    STD_COLUMNS = ["ball_speed", "mean_speed", "delta_d"]
    SLOW_COLUMNS = ["cent_H_x_hp", "cent_A_x_hp", "spread_H", "spread_A", "line_back_H", "line_front_H", "line_back_A",
                    "line_front_A", "length_H", "length_A"]
    # (columns, window half-lengths in seconds, statistics)
    GROUPS = [(CONTEXT_COLUMNS, (4, 12), ("mean", "trend")),
              (STD_COLUMNS, (4, 12), ("std",)),
              (SLOW_COLUMNS, (4, 12, 30), ("trend",))]

    def __init__(self, radius: int = 25, lags: Tuple[int, ...] = (12, 25), fps: int = 25):
        self.radius, self.lags, self.fps = radius, tuple(lags), fps

    def _half(self, sub: pd.DataFrame) -> Dict[str, np.ndarray]:
        out: Dict[str, np.ndarray] = {}
        keys = [c for c in self.KEY_COLUMNS if c in sub.columns]
        s = sub[keys].astype("float32").reset_index(drop=True)
        roll = s.rolling(2 * self.radius + 1, center=True, min_periods=1)
        for suf, df in (("mean", roll.mean()), ("std", roll.std()), ("min", roll.min()), ("max", roll.max())):
            for c in keys:
                out[f"{c}__{suf}"] = df[c].to_numpy(np.float32)
        for k in self.lags:
            for c in keys:
                out[f"{c}__lead{k}"] = (s[c].shift(-k) - s[c]).to_numpy(np.float32)
                out[f"{c}__lag{k}"] = (s[c].shift(k) - s[c]).to_numpy(np.float32)
        for columns, seconds, stats in self.GROUPS:
            present = [c for c in columns if c in sub.columns]
            if not present:
                continue
            A = sub[present].to_numpy(np.float64)
            for sec in seconds:
                w = int(round(sec * self.fps))
                if "mean" in stats or "std" in stats:
                    mean, sd = _roll(A, -w, w, want_std="std" in stats)
                    for j, c in enumerate(present):
                        if "mean" in stats:
                            out[f"{c}__mean{sec}s"] = mean[:, j].astype(np.float32)
                        if "std" in stats:
                            out[f"{c}__std{sec}s"] = sd[:, j].astype(np.float32)
                if "trend" in stats:
                    trend = _roll(A, 1, w)[0] - _roll(A, -w, -1)[0]
                    for j, c in enumerate(present):
                        out[f"{c}__trend{sec}s"] = trend[:, j].astype(np.float32)
        return out

    def transform(self, F: pd.DataFrame, half_blocks) -> pd.DataFrame:
        T = len(F)
        cols: Dict[str, np.ndarray] = {}
        for _, a, b in half_blocks:
            for name, arr in self._half(F.iloc[a:b + 1]).items():
                if name not in cols:
                    cols[name] = np.full(T, np.nan, np.float32)
                cols[name][a:b + 1] = arr
        if not cols:
            return model_view(F)
        return model_view(pd.concat([F.reset_index(drop=True), pd.DataFrame(cols)], axis=1))


In [ ]:
def softmax(z: np.ndarray, axis: int = -1) -> np.ndarray:
    z = z - np.nanmax(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def fit_temperature(logits: np.ndarray, y: np.ndarray) -> float:
    """Single temperature T minimising the NLL of softmax(logits / T) on held-out frames."""
    m = np.isfinite(logits).all(axis=1) & (y >= 0)
    if m.sum() < 50:
        return 1.0
    nll = lambda T: -np.mean(np.log(np.clip(softmax(logits[m] / T)[np.arange(m.sum()), y[m]], 1e-9, 1)))
    return float(minimize_scalar(nll, bounds=(0.3, 6.0), method="bounded").x)


CATBOOST_BASE = dict(loss_function="MultiClass", depth=6, l2_leaf_reg=5.0, border_count=64, rsm=0.5, thread_count=-1, verbose=False,
                     allow_writing_files=False)


class PossessionClassifier:
    """Frame-level CatBoost classifier -> log-probabilities / calibrated probabilities over [H, A, D]."""

    def __init__(self, seed: int = 0, temperature: float = 1.0, params: Optional[dict] = None):
        self.seed, self.temperature, self.params = seed, temperature, params or {}
        self.model_, self.columns_, self.best_iteration_ = None, None, None

    def _params(self, **over) -> dict:
        return {**CATBOOST_BASE, "random_seed": self.seed, **self.params, **over}

    def _np(self, X) -> np.ndarray:
        if isinstance(X, pd.DataFrame):
            X = X[self.columns_] if self.columns_ is not None else X
            return X.to_numpy(np.float32)
        return np.asarray(X, np.float32)

    def fit(self, X: pd.DataFrame, y: np.ndarray, w: Optional[np.ndarray] = None, eval_set=None,
            iterations: int = 1500, learning_rate: float = 0.08, early_stopping_rounds: int = 100) -> "PossessionClassifier":
        """Train from scratch. With eval_set=(X_val, y_val) training stops once the validation loss stops improving."""
        self.columns_ = list(X.columns)
        m = y >= 0
        kw = self._params(iterations=iterations, learning_rate=learning_rate)
        fit_kw = {}
        if eval_set is not None:
            Xv, yv = eval_set
            mv = yv >= 0
            kw.update(od_type="Iter", od_wait=early_stopping_rounds)
            fit_kw = dict(eval_set=(self._np(Xv)[mv], yv[mv].astype(int)), use_best_model=True)
        self.model_ = CatBoostClassifier(**kw)
        self.model_.fit(self._np(X)[m], y[m].astype(int), sample_weight=None if w is None else w[m], **fit_kw)
        best = self.model_.get_best_iteration() if eval_set is not None else None
        self.best_iteration_ = int(best) if best is not None else int(self.model_.tree_count_) - 1
        return self

    def predict_logits(self, X: pd.DataFrame, rows: Optional[np.ndarray] = None) -> np.ndarray:
        """(T, 3) log-probabilities; NaN where ``rows`` excludes a frame."""
        T = len(X)
        rows = np.arange(T) if rows is None else np.asarray(rows)
        out = np.full((T, 3), np.nan, np.float32)
        p = self.model_.predict_proba(self._np(X)[rows])
        full = np.full((len(rows), 3), 1e-6)
        full[:, np.asarray(self.model_.classes_, dtype=int)] = p
        full /= full.sum(1, keepdims=True)
        out[rows] = np.log(full)
        return out

    def predict_proba(self, X: pd.DataFrame, rows: Optional[np.ndarray] = None) -> np.ndarray:
        lg = self.predict_logits(X, rows)
        out = np.full_like(lg, np.nan)
        ok = np.isfinite(lg).all(axis=1)
        out[ok] = softmax(lg[ok] / self.temperature)
        return out

In [ ]:
def frame_scores(probs: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    """Per-frame accuracy / macro-F1 / log-loss on the rows with a label and finite probabilities."""
    m = (y >= 0) & np.isfinite(probs).all(axis=1)
    p, yt = probs[m].astype(np.float64), y[m].astype(int)
    p /= p.sum(axis=1, keepdims=True)
    return {"frames": int(m.sum()), "accuracy": round(float((p.argmax(1) == yt).mean()), 4),
            "macroF1": round(float(f1_score(yt, p.argmax(1), average="macro")), 4),
            "logloss": round(float(log_loss(yt, np.clip(p, 1e-9, 1), labels=[0, 1, 2])), 4)}

In [ ]:
import dataclasses
import hashlib

MODEL_DIR = PIPELINE_DIR / "models"


def _cfg_from_dict(cls, d: dict):
    """A config dataclass from a JSON dict (JSON turns tuples into lists)."""
    kw = {}
    for f in dataclasses.fields(cls):
        if f.name in d:
            v = d[f.name]
            kw[f.name] = tuple(v) if isinstance(f.default, tuple) and isinstance(v, list) else v
    return cls(**kw)


def _json_default(o):
    return float(o) if isinstance(o, (np.floating, np.integer)) else str(o)


def _code_hash() -> str:
    """Fingerprint of the notebooks that define preprocessing, features, model and decoder: tells load_bundle that the code changed since training."""
    h = hashlib.sha1()
    for f in ("helper_preprocessing.ipynb", "helper_features.ipynb", "helper_model.ipynb", "helper_decoder.ipynb"):
        for c in json.loads((PIPELINE_DIR / f).read_text(encoding="utf-8"))["cells"]:
            if c["cell_type"] == "code" and "defs" in c.get("metadata", {}).get("tags", []):
                h.update("".join(c["source"]).encode("utf-8"))
    return h.hexdigest()[:12]


def save_bundle(name: str, clf: "PossessionClassifier", dec_cfg: "HMMConfig", feat_cfg: "FeatureConfig", pre_cfg: "PreprocessConfig",
                info: Optional[dict] = None, root: Optional[Path] = None) -> Path:
    """Everything needed to label another match without retraining: the CatBoost model, the feature columns, the temperature, the tuned decoder,
    and the feature / preprocessing settings. -> models/<name>/ (catboost.cbm + bundle.json)"""
    d = (root or MODEL_DIR) / name
    d.mkdir(parents=True, exist_ok=True)
    clf.model_.save_model(str(d / "catboost.cbm"))
    meta = {"saved": time.strftime("%Y-%m-%d %H:%M:%S"), "columns": list(clf.columns_), "temperature": float(clf.temperature),
            "trees": int(clf.model_.tree_count_), "dec_cfg": asdict(dec_cfg), "feat_cfg": asdict(feat_cfg), "pre_cfg": asdict(pre_cfg),
            "info": info or {}, "code_hash": _code_hash()}
    (d / "bundle.json").write_text(json.dumps(meta, indent=1, default=_json_default), encoding="utf-8")
    return d


def load_bundle(name: str, root: Optional[Path] = None) -> dict:
    """-> {"clf": PossessionClassifier, "dec_cfg": HMMConfig, "feat_cfg": FeatureConfig, "pre_cfg": PreprocessConfig, "meta": dict}"""
    d = (root or MODEL_DIR) / name
    meta = json.loads((d / "bundle.json").read_text(encoding="utf-8"))
    clf = PossessionClassifier(temperature=meta["temperature"])
    clf.model_ = CatBoostClassifier().load_model(str(d / "catboost.cbm"))
    clf.columns_ = list(meta["columns"])
    clf.best_iteration_ = int(clf.model_.tree_count_) - 1
    if meta.get("code_hash") != _code_hash():
        print("WARNING: the preprocessing / feature / model / decoder code changed since this model was trained; retrain if the features differ.")
    return {"clf": clf, "dec_cfg": _cfg_from_dict(HMMConfig, meta["dec_cfg"]), "feat_cfg": _cfg_from_dict(FeatureConfig, meta["feat_cfg"]),
            "pre_cfg": _cfg_from_dict(PreprocessConfig, meta["pre_cfg"]), "meta": meta}


def predict_bundle(bundle: dict, clean: "CleanTracking") -> dict:
    """Label a match from its cleaned tracking with a saved model: features -> probabilities -> decoder. -> {"labels": (T,) '<U1', "probs": (T, 3)}"""
    X = WindowFeatureBuilder().transform(SpatialFeatureExtractor(bundle["feat_cfg"]).tabular(clean), clean.half_blocks)
    missing = [c for c in bundle["clf"].columns_ if c not in X.columns]
    if missing:
        raise ValueError(f"the feature code no longer produces {missing[:5]}...: the saved model was trained with different features")
    probs = bundle["clf"].predict_proba(X, rows=np.flatnonzero(clean.half_id() > 0))
    labels = HMMSequenceDecoder(bundle["dec_cfg"]).decode_match(probs, clean.half_blocks, clean.known_dead_ranges).labels
    return {"labels": labels, "probs": probs}